# MarketPulse: 01 - Exploratory Data Analysis & Temporal Dynamics

This notebook conducts research-grade exploratory data analysis on the M5 e-commerce demand dataset.

### Objectives:
1. **Sales Distribution Diagnostics**: Mean, median, dispersion, and zero-demand sparsity.
2. **Temporal Dynamics**: Day-of-week seasonality, monthly seasonality, and long-term trend.
3. **Exogenous Driver Analysis**: Holiday and SNAP (Supplemental Nutrition Assistance Program) uplift.
4. **Hierarchical Aggregations**: Demand roll-up across State, Store, Category, and Total business volume.

In [ ]:
import sys
from pathlib import Path

# Set project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.config import load_config
from src.data.hierarchy import M5Hierarchy
from src.features.temporal_features import calculate_seasonal_indices, calculate_event_uplift

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

config = load_config(project_root / "configs" / "config.yaml")
data_path = config.get_processed_path(config.data.processed_file)
print(f"Loading processed data from: {data_path}")
df = pd.read_parquet(data_path)
df.info()

## 1. Sales Distribution & Zero-Demand Sparsity

E-commerce daily retail demand exhibits extreme positive skewness and high zero-demand frequency.

In [ ]:
sales = df["sales"].dropna()
non_zero = sales[sales > 0]
zero_ratio = (sales == 0).mean() * 100

print(f"Total Observations: {len(sales):,}")
print(f"Mean Demand: {sales.mean():.2f} units/day (Median: {sales.median():.2f})")
print(f"Standard Deviation: {sales.std():.2f}")
print(f"Zero-Demand Percentage: {zero_ratio:.2f}%")
print(f"Non-Zero Mean Demand: {non_zero.mean():.2f} units/day (Non-Zero Std: {non_zero.std():.2f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(non_zero[non_zero <= non_zero.quantile(0.99)], bins=30, kde=True, ax=axes[0], color="#2980b9")
axes[0].set_title("Non-Zero Daily Sales Distribution (Truncated 99th Pct)")

sns.kdeplot(np.log1p(sales), ax=axes[1], fill=True, color="#8e44ad")
axes[1].set_title("Log-Transformed Daily Sales: ln(Sales + 1)")
plt.tight_layout()
plt.show()

## 2. Temporal Dynamics: Daily Trend & Day-of-Week Seasonality

In [ ]:
daily_total = df.groupby("date", observed=True)["sales"].sum().reset_index()
daily_total["rolling_7d"] = daily_total["sales"].rolling(7, min_periods=1).mean()

plt.figure(figsize=(14, 5))
plt.plot(daily_total["date"], daily_total["sales"], label="Daily Actuals", color="#3498db", alpha=0.6)
plt.plot(daily_total["date"], daily_total["rolling_7d"], label="7-Day Rolling Average", color="#e67e22", lw=2)
plt.title("Total Daily Business Demand Trend")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.show()

In [ ]:
seasonal_indices = calculate_seasonal_indices(df)
dow = seasonal_indices["dow_profile"]

plt.figure(figsize=(10, 4))
bars = plt.bar(dow["day_name"], dow["seasonal_index"], color="#1abc9c")
plt.axhline(1.0, color="red", linestyle="--", label="Neutral Baseline (1.0)")
plt.title("Day-of-Week Seasonality Index")
plt.ylabel("Seasonality Index")
for b in bars:
    y = b.get_height()
    plt.text(b.get_x() + b.get_width()/2, y + 0.02, f"{y:.2f}", ha="center")
plt.legend()
plt.show()

## 3. Exogenous Effects: SNAP and Holiday Uplifts

In [ ]:
uplift = calculate_event_uplift(df)
print("SNAP Benefit Impact:")
for k, v in uplift["snap"].items():
    print(f"  {k}: {v}")

if not uplift["events"].empty:
    print("\nTop Event Uplifts:")
    display(uplift["events"].sort_values("uplift_ratio", ascending=False).head(10))


## 4. Hierarchical Demand Roll-Up

In [ ]:
hierarchy = M5Hierarchy(metadata_df=df)
state_demand = hierarchy.aggregate_bottom_up(df, level=2)
cat_demand = hierarchy.aggregate_bottom_up(df, level=4)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for st in state_demand["series_id"].unique():
    sub = state_demand[state_demand["series_id"] == st]
    axes[0].plot(sub["date"], sub["sales"], label=f"State {st}")
axes[0].set_title("State-Level Demand Roll-Up")
axes[0].legend()

for cat in cat_demand["series_id"].unique():
    sub = cat_demand[cat_demand["series_id"] == cat]
    axes[1].plot(sub["date"], sub["sales"], label=cat)
axes[1].set_title("Category-Level Demand Roll-Up")
axes[1].legend()
plt.tight_layout()
plt.show()